# ChemML data acquisition and database inspection

This introductory notebook demonstrates the implemented ChemML database-update workflow. It loads the configuration for a region, requests a bounded interval from the enabled input-data sources, and then reads and visualizes the resulting partitioned databases.

The example interval is **August 4–7, 2026** (the end is exclusive). Change `CONFIG_PATH`, `START_TIME`, and `END_TIME` in the setup cell when adapting this notebook to another region or period. The committed example points to `config/default_config.yaml`; local testing can use `config/testing_config.yaml`. Remote credentials and network access are required for sources whose databases are not already complete.

## 1. Initial setup

The first code cell imports the high-level entry point, the configuration loader, the database manager, and the shared plotting helper. It also defines the region/configuration and time interval used throughout the notebook.

In [ ]:
# Activate the chemml Conda environment before starting Jupyter.
import os
os.chdir('/path/to/ChemML') # Update this path to reflect the ChemML directory on your machine.
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from aqdatafusion import run_aqdatafusion
from aqdatafusion.config_management.configuration import load_configuration
from aqdatafusion.data_management import DatabaseManager
from aqdatafusion.utils import plot_data_map

# The committed example uses the repository's general configuration.
CONFIG_PATH = Path("config/default_config.yaml")
# For local, controlled testing, use the ATL testing configuration instead:
# CONFIG_PATH = Path("config/testing_config.yaml")

# The interval is half-open: data at START_TIME are included, while data
# at END_TIME are not. Change these values for another demonstration.
START_TIME = "2026-08-04T00:00:00Z"
END_TIME = "2026-08-07T00:00:00Z"

configuration = load_configuration(CONFIG_PATH, require_credentials=False)
region = configuration["region"]
print(f"Region: {region['name']} ({region['region_code']})")
print(f"Interval: {START_TIME} to {END_TIME}")

## 2. Run a bounded database update

`run_aqdatafusion` is the high-level entry point. In database-update mode it checks the existing partitions, acquires only missing data, preprocesses it, writes the configured partitions, rebuilds source catalogs, and performs integrity checks. Setting `require_credentials=True` makes missing or invalid remote credentials fail clearly before acquisition begins.

In [ ]:
# Update every enabled source in the selected configuration.
SOURCE_KEYS = None
# To restrict the update, replace None with one of these examples:
# SOURCE_KEYS = ["ground_monitors"]
# SOURCE_KEYS = ["GEOS-CF"]
# SOURCE_KEYS = ["TROPOMI", "VIIRS", "land_cover"]
update_result = run_aqdatafusion(
    CONFIG_PATH,
    require_credentials=True,
    mode="database_update",
    start_time=START_TIME,
    end_time=END_TIME,
    source_keys=SOURCE_KEYS,
)


print(update_result["status"])
print(update_result["message"])
if update_result.get("credentials"):
    credential_check = update_result["credentials"]
    print("Credential verification:")
    for provider, verification in credential_check.get("verification", {}).items():
        print(f"  {provider}: {verification.get('status')} — {verification.get('message')}")
    if credential_check.get("errors"):
        print("Credential errors:", credential_check["errors"])
if "report" in update_result:
    report = update_result["report"]
    print(report.get("operational_report_path", ""))
    print("Source statuses:")
    for source_key, source_report in report.get("sources", {}).items():
        print(f"  {source_key}: {source_report.get('status')}")

The optional `source_keys` argument is useful when developing or diagnosing one adapter. Omitting it uses the sources marked `enabled: true` in `aqdatafusion_config.yaml`. The update report contains per-source status, missing timestamps, written files, integrity findings, and wall-time measurements.

## 3. Read and summarize the current databases

The public `DatabaseManager.read_database()` method reads the current on-disk database for one configured source. Point-monitor data are returned as a pandas DataFrame; gridded data are returned as a lazily opened xarray Dataset.

In [ ]:
manager = DatabaseManager(configuration)
enabled_sources = [
    name for name, settings in configuration["aqdatafusion"]["inputs"].items()
    if settings.get("enabled", False)
]

loaded_data = {}
for source_key in enabled_sources:
    data = manager.read_database(source_key)
    loaded_data[source_key] = data
    print(f"\n{source_key}")
    if isinstance(data, pd.DataFrame):
        print(f"  type: pandas DataFrame, rows={len(data):,}, columns={list(data.columns)}")
        if "time" in data and len(data):
            print(f"  time: {data['time'].min()} to {data['time'].max()}")
    elif data is None:
        print("  no database files found")
    else:
        print(data)
        print(f"  time: {str(data.time.values.min())[0:16]} to {str(data.time.values.max())[0:16]}")

## 4. Time-mean maps

The shared `plot_data_map` function accepts either an xarray object or point-data DataFrame. It uses the configured regional extent and the Cartopy cache created by first-time setup. If base layers are unavailable offline, the data and gridlines are still plotted without those optional layers.

In [ ]:
# Select a useful variable from each loaded source and show its time mean.
for source_key, data in loaded_data.items():
    if data is None:
        continue
    if isinstance(data, pd.DataFrame):
        candidates = [name for name in ["PM2.5", "NO2", "O3"] if name in data]
        if not candidates:
            continue
        variable = candidates[0]
        figure, _ = plot_data_map(
            data, variable=variable, configuration=configuration,
            title=f"{source_key}: mean {variable}", colorbar_label=variable,
        )
    else:
        variables = list(data.data_vars)
        if not variables:
            continue
        variable = variables[0]
        figure, _ = plot_data_map(
            data, variable=variable, configuration=configuration,
            title=f"{source_key}: time mean {variable}", colorbar_label=variable,
        )
    plt.show()

## 5. Compare sources at one timestamp

A single timestamp is useful for checking spatial coverage and for comparing the different temporal resolutions. Satellite products may not have observations at every hour, so the plotting helper uses the nearest available time for gridded data.

In [ ]:
TIMESTAMP = "2026-08-04T15:00:00Z"  # Change this to inspect another time.
for source_key, data in loaded_data.items():
    if data is None:
        continue
    variable = (
        next((name for name in ["PM2.5", "NO2", "O3"] if name in data), None)
        if isinstance(data, pd.DataFrame)
        else (list(data.data_vars)[0] if data.data_vars else None)
    )
    if variable is None:
        continue
    plot_data_map(
        data, variable=variable, timestamp=TIMESTAMP, configuration=configuration,
        title=f"{source_key}: {variable} near {TIMESTAMP}", colorbar_label=variable,
    )
    plt.show()

## 6. Temporal summaries

Finally, the following cells illustrate simple temporal summaries. These are deliberately kept outside the acquisition workflow: the database stores source data, while analysis choices such as spatial averaging, resampling, and variable selection belong to the user.

In [ ]:
# Plot a time series for the first available variable from each source.
fig, ax = plt.subplots(figsize=(12, 5))
for source_key, data in loaded_data.items():
    if isinstance(data, pd.DataFrame):
        variable = next((name for name in ["PM2.5", "NO2", "O3"] if name in data), None)
        if variable is None or "time" not in data:
            continue
        series = data.assign(time=pd.to_datetime(data["time"], utc=True)).set_index("time")[variable].resample("1h").mean()
    elif data is not None and "time" in data.dims:
        variable = list(data.data_vars)[0]
        data_array = data[variable]
        # GEOS-CF has a forecast_lead_day dimension. Select analysis
        # (lead day 0) before reducing the spatial dimensions.
        extra_dims = [dim for dim in data_array.dims if dim not in {"time", "lat", "lon"}]
        if extra_dims:
            data_array = data_array.isel({dim: 0 for dim in extra_dims})
        spatial_dims = [dim for dim in ["lat", "lon"] if dim in data_array.dims]
        series = data_array.mean(dim=spatial_dims, skipna=True).to_series()
        series.index = pd.to_datetime(series.index, utc=True)
    else:
        continue
    series = series.dropna()
    if series.empty:
        continue
    minimum, maximum = series.min(), series.max()
    normalized = (series - minimum) / (maximum - minimum) if maximum > minimum else series * 0.0
    normalized.plot(ax=ax, label=source_key)
ax.set_title("Database temporal coverage (each dataset normalized to 0–1)")
ax.set_xlabel("UTC time")
ax.set_ylabel("Min–max normalized value")
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Close lazily opened xarray datasets when the inspection is complete.
for data in loaded_data.values():
    close = getattr(data, "close", None)
    if callable(close):
        close()